# Диагностика статической обратной задачи

Ноутбук проверяет локальную определимость и устойчивость результата `33.01`.
Он не создаёт альтернативную «лучшую» оценку и не трактует 36 или 45
попарных решений как независимые наблюдения.

Реальная диагностика запускается только по внешнему артефакту
`33.01_static.json`. Без него выполняется синтетическая самопроверка кода.


## Диагностические проверки

Для совместной оценки $(\rho_1,\rho_{2,\mathrm{вд}},
\rho_{2,\mathrm{выд}})$ рассчитываются сингулярные числа логарифмического
Якобиана, его ранг и число обусловленности. Затем выполняются:

- профиль невязки по $\rho_{2,\mathrm{вд}}$ с повторной оптимизацией остальных
  параметров;
- исключение одного размера целиком, то есть сразу двух дыхательных
  наблюдений;
- проверка выхода решения к численным границам и систематики остатка по $L$.

Без внешней ковариации межзаписной ошибки не вычисляются доверительные
интервалы, статистическая значимость и CRLB. Внутриплатовый разброс не заменяет
межзаписную ковариацию.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import numpy as np
from scipy.optimize import least_squares

from two_layer_model import evaluate, geometry_from_size

REAL_MODE = os.environ.get("KALMYKOV_RUN_REAL", "0") == "1"


def predict(theta_log, sizes_m, h_m):
    rho1, rho2_in, rho2_ex = np.exp(np.asarray(theta_log, dtype=float))
    values = []
    rows = []
    for size_m in sizes_m:
        a, b = geometry_from_size(float(size_m))
        inhale = evaluate(rho1, rho2_in, h_m, a, b)
        exhale = evaluate(rho1, rho2_ex, h_m, a, b)
        values.extend([inhale.z, exhale.z])
        rows.extend([
            [inhale.d_rho1 * rho1, inhale.d_rho2 * rho2_in, 0.0],
            [exhale.d_rho1 * rho1, 0.0, exhale.d_rho2 * rho2_ex],
        ])
    return np.asarray(values), np.asarray(rows)


def refit(sizes_m, observed, h_m, initial=None, fixed_rho2_in=None):
    sizes_m = np.asarray(sizes_m, dtype=float)
    observed = np.asarray(observed, dtype=float)
    if fixed_rho2_in is None:
        seed = np.log([5.0, 20.0, 15.0]) if initial is None else np.log(initial)
        objective = lambda x: predict(x, sizes_m, h_m)[0] - observed
        solution = least_squares(objective, seed)
        theta_log = solution.x
    else:
        seed = np.log([5.0, 15.0]) if initial is None else np.log([initial[0], initial[2]])
        def objective(x):
            theta = np.asarray([x[0], np.log(fixed_rho2_in), x[1]])
            return predict(theta, sizes_m, h_m)[0] - observed
        solution = least_squares(objective, seed)
        theta_log = np.asarray([solution.x[0], np.log(fixed_rho2_in), solution.x[1]])
    predicted, jacobian = predict(theta_log, sizes_m, h_m)
    return np.exp(theta_log), predicted - observed, jacobian


def diagnose(sizes_m, z_in, z_ex, h_m, estimate):
    sizes_m = np.asarray(sizes_m, dtype=float)
    observed = np.column_stack([z_in, z_ex]).reshape(-1)
    theta, residual, jacobian = refit(sizes_m, observed, h_m, initial=estimate)
    singular = np.linalg.svd(jacobian, compute_uv=False)
    profile_grid = np.geomspace(max(0.25 * theta[1], 0.1), 4.0 * theta[1], 31)
    profile_rms = []
    for value in profile_grid:
        _, res, _ = refit(sizes_m, observed, h_m, initial=theta, fixed_rho2_in=float(value))
        profile_rms.append(float(np.sqrt(np.mean(res ** 2))))
    leave_one_out = []
    for index, size_m in enumerate(sizes_m):
        keep = np.ones(len(sizes_m), dtype=bool)
        keep[index] = False
        obs_keep = np.column_stack([np.asarray(z_in)[keep], np.asarray(z_ex)[keep]]).reshape(-1)
        loo_theta, loo_res, _ = refit(sizes_m[keep], obs_keep, h_m, initial=theta)
        leave_one_out.append({
            "removed_size_mm": float(size_m * 1000.0),
            "estimate": loo_theta.tolist(),
            "rms_ohm": float(np.sqrt(np.mean(loo_res ** 2))),
        })
    return {
        "estimate": theta.tolist(),
        "rms_ohm": float(np.sqrt(np.mean(residual ** 2))),
        "rank": int(np.linalg.matrix_rank(jacobian)),
        "singular_values": singular.tolist(),
        "condition": float(singular[0] / singular[-1]) if singular[-1] > 0 else float("inf"),
        "profile_rho2_in_ohm_m": profile_grid.tolist(),
        "profile_rms_ohm": profile_rms,
        "leave_one_size_out": leave_one_out,
        "residual_ohm": residual.tolist(),
    }


In [ ]:
sizes_test = np.asarray([0.05, 0.06, 0.07, 0.08, 0.09, 0.11, 0.12, 0.13, 0.14])
h_test = 0.020
truth = np.asarray([5.0, 18.0, 14.0])
predicted_test, _ = predict(np.log(truth), sizes_test, h_test)
z_pair_test = predicted_test.reshape(-1, 2)
diagnostic_test = diagnose(sizes_test, z_pair_test[:, 0], z_pair_test[:, 1], h_test, truth)
assert diagnostic_test["rank"] == 3
assert len(diagnostic_test["leave_one_size_out"]) == len(sizes_test)
assert diagnostic_test["rms_ohm"] < 1e-9
profile_min = int(np.argmin(diagnostic_test["profile_rms_ohm"]))
assert 0 < profile_min < len(diagnostic_test["profile_rms_ohm"]) - 1
print("33.02 synthetic_self_test: passed")


In [ ]:
if not REAL_MODE:
    print("33.02 real_data_status: blocked_until_33.01_external_artifact_exists")
else:
    config_value = os.environ.get("KALMYKOV_EXP02_CONFIG")
    if not config_value:
        raise RuntimeError("Задайте KALMYKOV_EXP02_CONFIG")
    config = json.loads(Path(config_value).expanduser().resolve().read_text(encoding="utf-8"))
    derived_root = Path(config["derived_root"]).expanduser().resolve()
    static_path = derived_root / "exp02" / "analysis" / "33.01_static.json"
    static = json.loads(static_path.read_text(encoding="utf-8"))
    if static.get("status") != "conditional_two_layer_estimate":
        raise RuntimeError("33.01 не имеет требуемого статуса")
    results = {}
    for subject_id, item in static["subjects"].items():
        estimate = item["estimate"]
        initial = [estimate["rho1_ohm_m"], estimate["rho2_inhale_ohm_m"], estimate["rho2_exhale_ohm_m"]]
        results[subject_id] = diagnose(
            np.asarray(item["sizes_mm"], dtype=float) / 1000.0,
            item["z_inhale_ohm"], item["z_exhale_ohm"], item["h_m"], initial,
        )
    output = {
        "schema_version": 1,
        "analysis": "33.02_static_diagnostics",
        "status": "diagnostic_without_error_covariance",
        "source_33_01": str(static_path.name),
        "subjects": results,
        "limitations": [
            "no_inter_record_error_covariance",
            "leave_one_size_out_is_not_independent_validation",
            "pairwise_solutions_are_not_independent_observations",
        ],
    }
    out_path = derived_root / "exp02" / "analysis" / "33.02_diagnostics.json"
    out_path.write_text(json.dumps(output, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print("33.02 real_data_status: diagnostic_written", out_path)


## Критерий завершения

Расчёт считается технически выполненным, если профиль имеет внутренний минимум,
логарифмический Якобиан имеет полный локальный ранг, а исключение одного размера
не переводит решение на существенно иной участок пространства параметров.
Эти признаки не доказывают правильность двуслойной физической модели.
Независимая проверка требует КТ/FEM, калибровки и заранее заданного бюджета
погрешности.
